In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install wfdb scipy numpy pandas openpyxl xlrd scikit-learn tqdm mne -q

In [ ]:
import os
import wfdb
import mne
import numpy as np
import pandas as pd
from scipy import signal as scipy_signal
from scipy.signal import butter, filtfilt, resample
import pickle
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ---- PATHS (Update these to your local paths) ----
PHYSIONET_DIR = '/content/datasets/physionet/apnea-ecg-database-1.0.0'
UCDDB_DIR     = '/content/drive/MyDrive/sleep_apnea/files'
OUTPUT_DIR    = '/content/drive/MyDrive/sleep_apnea/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Signal Parameters ----
EPOCH_DURATION    = 30      # seconds per epoch (PSG standard)
PHYSIONET_FS      = 100     # PhysioNet ECG sampling rate (Hz)
UCDDB_FS          = 128     # UCDDB ECG sampling rate (Hz)
TARGET_FS         = 100     # Unified target sampling rate
ECG_SAMPLES       = EPOCH_DURATION * TARGET_FS   # 3000 samples per epoch

print(f'PhysioNet DIR  : {PHYSIONET_DIR}')
print(f'UCDDB DIR      : {UCDDB_DIR}')
print(f'Output DIR     : {OUTPUT_DIR}')
print(f'Epoch length   : {EPOCH_DURATION}s = {ECG_SAMPLES} samples @ {TARGET_FS}Hz')

Extracting UCDDB database...
Extraction complete.
UCDDB DIR      : /content/ucddb_data/files
Output DIR     : /content/processed


In [ ]:
import os
import zipfile

# ---- ZIP Paths ----
ZIP_PHYSIONET = '/content/drive/MyDrive/SA ECG dataset/apnea-ecg-database-1.0.0.zip'
ZIP_UCDDB = '/content/drive/MyDrive/SA ECG dataset/st-vincents-university-hospital-university-college-dublin-sleep-apnea-database-1.0.0.zip'

# ---- Extraction Destinations ----
PHYSIONET_EXTRACT_DIR = '/content/physionet_data'
UCDDB_EXTRACT_DIR = '/content/ucddb_data'

# Extract PhysioNet
if os.path.exists(ZIP_PHYSIONET):
    print("Extracting PhysioNet database...")
    with zipfile.ZipFile(ZIP_PHYSIONET, 'r') as zip_ref:
        zip_ref.extractall(PHYSIONET_EXTRACT_DIR)
    print("PhysioNet extraction complete.")
else:
    print(f"PhysioNet ZIP not found at: {ZIP_PHYSIONET}")

# Extract UCDDB
if os.path.exists(ZIP_UCDDB):
    print("Extracting UCDDB database...")
    with zipfile.ZipFile(ZIP_UCDDB, 'r') as zip_ref:
        zip_ref.extractall(UCDDB_EXTRACT_DIR)
    print("UCDDB extraction complete.")
else:
    print(f"UCDDB ZIP not found at: {ZIP_UCDDB}")

# ---- Updated Path Configurations ----
PHYSIONET_DIR = os.path.join(PHYSIONET_EXTRACT_DIR, 'apnea-ecg-database-1.0.0')
UCDDB_DIR     = os.path.join(UCDDB_EXTRACT_DIR, 'files')
OUTPUT_DIR    = '/content/processed'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'\nUpdated Paths:')
print(f'PhysioNet DIR  : {PHYSIONET_DIR}')
print(f'UCDDB DIR      : {UCDDB_DIR}')
print(f'Output DIR     : {OUTPUT_DIR}')

Extracting PhysioNet database...
PhysioNet extraction complete.
Extracting UCDDB database...
UCDDB extraction complete.

Updated Paths:
PhysioNet DIR  : /content/physionet_data/apnea-ecg-database-1.0.0
UCDDB DIR      : /content/ucddb_data/files
Output DIR     : /content/processed


In [ ]:
def bandpass_filter(signal_data, lowcut=0.5, highcut=40.0, fs=100, order=4):
    nyq = fs / 2.0
    low, high = lowcut / nyq, highcut / nyq
    high = min(high, 0.99)
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal_data)

def normalize_signal(sig):
    mu, sigma = np.mean(sig), np.std(sig)
    return (sig - mu) / (sigma + 1e-8)

def compute_sqi(epoch, fs=100):
    total_power = np.sum(epoch**2) + 1e-10
    filtered = bandpass_filter(epoch, 0.5, 40.0, fs)
    band_power = np.sum(filtered**2)
    sqi = np.clip(band_power / total_power, 0.0, 1.0)
    return float(sqi)

def resample_to_target(sig, orig_fs, target_fs, target_len):
    if orig_fs != target_fs:
        ratio = target_fs / orig_fs
        new_len = int(len(sig) * ratio)
        sig = resample(sig, new_len)
    if len(sig) >= target_len:
        return sig[:target_len]
    else:
        return np.pad(sig, (0, target_len - len(sig)), mode='constant')

In [ ]:
import os
import numpy as np
from tqdm.auto import tqdm
import wfdb
from scipy.signal import butter, filtfilt

# Required signal parameters
EPOCH_DURATION = 30
TARGET_FS      = 100
ECG_SAMPLES    = EPOCH_DURATION * TARGET_FS

PHYSIONET_LABELED_SUBJECTS = (
    [f'a{i:02d}' for i in range(1, 21)] +
    [f'b{i:02d}' for i in range(1, 6)]  +
    [f'c{i:02d}' for i in range(1, 11)]
)

def load_physionet_subject(subject_id, data_dir):
    record_path = os.path.join(data_dir, subject_id)
    apn_path    = os.path.join(data_dir, f'{subject_id}.apn')

    if not os.path.exists(record_path + '.dat') or not os.path.exists(apn_path):
        return []

    record = wfdb.rdrecord(record_path)
    ecg    = record.p_signal[:, 0].astype(np.float32)
    fs     = record.fs

    ann = wfdb.rdann(record_path, 'apn')
    labels_per_minute = ann.symbol

    epochs = []
    samples_per_minute = int(fs * 60)
    samples_per_epoch  = int(fs * EPOCH_DURATION)

    for min_idx, label_char in enumerate(labels_per_minute):
        label = 1 if label_char == 'A' else 0
        for half in range(2):
            start = min_idx * samples_per_minute + half * samples_per_epoch
            end   = start + samples_per_epoch
            if end > len(ecg): break

            epoch_ecg = ecg[start:end].copy()
            epoch_ecg = bandpass_filter(epoch_ecg, 0.5, 40.0, fs)
            epoch_ecg = resample_to_target(epoch_ecg, fs, TARGET_FS, ECG_SAMPLES)
            sqi       = compute_sqi(epoch_ecg, TARGET_FS)
            epoch_ecg = normalize_signal(epoch_ecg)

            epochs.append({
                'subject_id' : subject_id,
                'epoch_idx'  : min_idx * 2 + half,
                'ecg'        : epoch_ecg.astype(np.float32),
                'label'      : label,
                'sqi'        : np.float32(sqi),
                'dataset'    : 'physionet',
                'group'      : subject_id[0],
            })
    return epochs

print('Processing PhysioNet Apnea-ECG dataset...')
physionet_data = []
for subj in tqdm(PHYSIONET_LABELED_SUBJECTS, desc='PhysioNet Subjects'):
    epochs = load_physionet_subject(subj, PHYSIONET_DIR)
    physionet_data.extend(epochs)

if physionet_data:
    pos_epochs = sum(e['label'] for e in physionet_data)
    print(f'\nPhysioNet Summary: {len(physionet_data)} epochs | Apnea: {pos_epochs}')

Processing PhysioNet Apnea-ECG dataset...


PhysioNet Subjects:   0%|          | 0/35 [00:00<?, ?it/s]


PhysioNet Summary: 34053 epochs | Apnea: 13023


In [ ]:
import mne
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from scipy.signal import butter, filtfilt, resample
def read_subject_details(xls_path):
    try:
        return pd.read_excel(xls_path, engine='xlrd')
    except:
        return pd.read_excel(xls_path, engine='openpyxl')

def parse_respevt(respevt_path):
    events = []
    try:
        with open(respevt_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            line = line.strip()
            if not line or line.startswith('#') or line.startswith('Time') or 'Respiratory' in line or 'Type' in line:
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    time_str = parts[0]
                    h, m, s  = map(int, time_str.split(':'))
                    start_sec = h * 3600 + m * 60 + s
                    evt_type = parts[1]
                    try:
                        duration = float(parts[2])
                    except ValueError:
                        duration = float(parts[3])

                    events.append({
                        'start_sec'  : start_sec,
                        'end_sec'    : start_sec + duration,
                        'duration'   : duration,
                        'event_type' : evt_type
                    })
                except Exception:
                    continue
    except Exception as e:
        print(f'    Warning parsing {respevt_path}: {e}')
    return events

def events_to_epoch_labels(events, recording_duration_sec, epoch_duration=30):
    n_epochs = int(recording_duration_sec // epoch_duration)
    labels   = np.zeros(n_epochs, dtype=np.int32)
    apnea_keywords = ['apnea', 'hyp', 'osa', 'csa']

    for evt in events:
        if not any(kw in evt['event_type'].lower() for kw in apnea_keywords):
            continue
        evt_start, evt_end = evt['start_sec'], evt['end_sec']

        for ep_idx in range(n_epochs):
            ep_start = ep_idx * epoch_duration
            ep_end   = ep_start + epoch_duration
            overlap  = max(0, min(evt_end, ep_end) - max(evt_start, ep_start))
            if overlap >= epoch_duration * 0.5:
                labels[ep_idx] = 1
    return labels

def load_ucddb_subject(subject_id, data_dir, clinical_df):
    rec_path  = os.path.join(data_dir, f'{subject_id}.rec')
    evnt_path = os.path.join(data_dir, f'{subject_id}_respevt.txt')

    if not os.path.exists(rec_path) or not os.path.exists(evnt_path):
        return []

    try:
        # Use MNE directly (NO .hea file needed)
        raw = mne.io.read_raw_edf(rec_path, preload=True, verbose=False)
        sig_names = [s.lower() for s in raw.ch_names]
        fs        = raw.info['sfreq']
        signals   = raw.get_data().T.astype(np.float32)
    except Exception as e:
        print(f'  [SKIP] {subject_id}: {e}')
        return []

    ecg_idx  = next((i for i, n in enumerate(sig_names) if 'ecg' in n), None)
    spo2_idx = next((i for i, n in enumerate(sig_names) if 'spo2' in n or 'sat' in n or 'sao2' in n), None)
    resp_idx = next((i for i, n in enumerate(sig_names) if 'resp' in n or 'thor' in n or 'abd' in n or 'flow' in n), None)

    if ecg_idx is None:
        return []

    ecg_raw  = signals[:, ecg_idx]
    spo2_raw = signals[:, spo2_idx] if spo2_idx is not None else np.zeros(len(ecg_raw))
    resp_raw = signals[:, resp_idx] if resp_idx is not None else np.zeros(len(ecg_raw))

    events   = parse_respevt(evnt_path)
    rec_dur  = len(ecg_raw) / fs
    labels   = events_to_epoch_labels(events, rec_dur, EPOCH_DURATION)
    clinical_feat = extract_clinical_features(subject_id, clinical_df)

    n_apnea_epochs = labels.sum()
    rec_hrs        = rec_dur / 3600.0
    ahi_estimate   = n_apnea_epochs / max(rec_hrs, 0.1)

    samples_per_epoch = int(fs * EPOCH_DURATION)
    epochs = []

    for ep_idx, label in enumerate(labels):
        start = ep_idx * samples_per_epoch
        end   = start + samples_per_epoch
        if end > len(ecg_raw): break

        ecg_ep  = ecg_raw[start:end].copy()
        spo2_ep = spo2_raw[start:end].copy()
        resp_ep = resp_raw[start:end].copy()

        ecg_ep  = bandpass_filter(ecg_ep, 0.5, 40.0, fs)
        ecg_ep  = resample_to_target(ecg_ep, fs, TARGET_FS, ECG_SAMPLES)
        spo2_ep = resample_to_target(spo2_ep, fs, TARGET_FS, ECG_SAMPLES)
        resp_ep = resample_to_target(resp_ep, fs, TARGET_FS, ECG_SAMPLES)

        sqi = compute_sqi(ecg_ep, TARGET_FS)

        ecg_ep  = normalize_signal(ecg_ep)
        spo2_ep = normalize_signal(spo2_ep)
        resp_ep = normalize_signal(resp_ep)

        multimodal = np.stack([ecg_ep, spo2_ep, resp_ep], axis=0).astype(np.float32)

        epochs.append({
            'subject_id': subject_id,
            'epoch_idx' : ep_idx,
            'ecg'       : ecg_ep.astype(np.float32),
            'multimodal': multimodal,
            'label'     : int(label),
            'ahi'       : np.float32(ahi_estimate),
            'sqi'       : np.float32(sqi),
            'clinical'  : clinical_feat,
            'dataset'   : 'ucddb',
        })
    return epochs

def extract_clinical_features(subject_id, clinical_df):
    # Clinical Features: Age, Sex/Gender, BMI, Neck Circ (default 38.0), AHI
    defaults = np.array([50.0, 0.5, 27.0, 38.0, 15.0], dtype=np.float32)
    if clinical_df is None: return defaults
    try:
        # Match case-insensitively (e.g. 'ucddb002' to 'UCDDB002')
        row = clinical_df[clinical_df.iloc[:, 1].astype(str).str.lower() == subject_id.lower()]
        if len(row) == 0: return defaults
        row = row.iloc[0]
        cols = clinical_df.columns.tolist()

        def safe_get(col_keywords, default):
            for kw in col_keywords:
                matched = [c for c in cols if kw.lower() in str(c).lower()]
                if matched:
                    val = row[matched[0]]
                    if pd.notna(val):
                        # Handle M/F gender translation
                        if str(val).strip().upper() == 'M':
                            return 1.0
                        elif str(val).strip().upper() == 'F':
                            return 0.0
                        return float(val)
            return float(default)

        return np.array([
            safe_get(['age'], 50.0),
            safe_get(['gender', 'sex'], 0.5),
            safe_get(['bmi'], 27.0),
            safe_get(['neck', 'collar'], 38.0),  # Neck circumference is not in XLS, so defaults to 38.0
            safe_get(['ahi', 'rdi', 'apnea'], 15.0)
        ], dtype=np.float32)
    except Exception:
        return defaults

In [ ]:
import mne
import mne
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from scipy.signal import butter, filtfilt, resample

def load_ucddb_subject(subject_id, data_dir, clinical_df):
    rec_path  = os.path.join(data_dir, f'{subject_id}.rec')
    evnt_path = os.path.join(data_dir, f'{subject_id}_respevt.txt')

    if not os.path.exists(rec_path) or not os.path.exists(evnt_path):
        return []

    try:
        # .rec is actually EDF format, just wrong extension
        # Create a symlink with .edf extension so MNE accepts it
        edf_path = os.path.join(data_dir, f'{subject_id}.edf')
        if not os.path.exists(edf_path):
            os.symlink(rec_path, edf_path)

        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        sig_names = [s.lower() for s in raw.ch_names]
        fs        = raw.info['sfreq']
        signals   = raw.get_data().T.astype(np.float32)
    except Exception as e:
        print(f'  [SKIP] {subject_id}: {e}')
        return []

    ecg_idx  = next((i for i, n in enumerate(sig_names) if 'ecg' in n), None)
    spo2_idx = next((i for i, n in enumerate(sig_names) if 'spo2' in n or 'sat' in n or 'sao2' in n), None)
    resp_idx = next((i for i, n in enumerate(sig_names) if 'resp' in n or 'thor' in n or 'abd' in n or 'flow' in n), None)

    if ecg_idx is None:
        print(f'  [SKIP] {subject_id}: No ECG. Channels: {sig_names}')
        return []

    ecg_raw  = signals[:, ecg_idx]
    spo2_raw = signals[:, spo2_idx] if spo2_idx is not None else np.zeros(len(ecg_raw))
    resp_raw = signals[:, resp_idx] if resp_idx is not None else np.zeros(len(ecg_raw))

    events   = parse_respevt(evnt_path)
    rec_dur  = len(ecg_raw) / fs
    labels   = events_to_epoch_labels(events, rec_dur, EPOCH_DURATION)
    clinical_feat = extract_clinical_features(subject_id, clinical_df)

    n_apnea_epochs = labels.sum()
    rec_hrs        = rec_dur / 3600.0
    ahi_estimate   = n_apnea_epochs / max(rec_hrs, 0.1)

    samples_per_epoch = int(fs * EPOCH_DURATION)
    epochs = []

    for ep_idx, label in enumerate(labels):
        start = ep_idx * samples_per_epoch
        end   = start + samples_per_epoch
        if end > len(ecg_raw): break

        ecg_ep  = ecg_raw[start:end].copy()
        spo2_ep = spo2_raw[start:end].copy()
        resp_ep = resp_raw[start:end].copy()

        ecg_ep  = bandpass_filter(ecg_ep, 0.5, 40.0, fs)
        ecg_ep  = resample_to_target(ecg_ep, fs, TARGET_FS, ECG_SAMPLES)
        spo2_ep = resample_to_target(spo2_ep, fs, TARGET_FS, ECG_SAMPLES)
        resp_ep = resample_to_target(resp_ep, fs, TARGET_FS, ECG_SAMPLES)

        sqi = compute_sqi(ecg_ep, TARGET_FS)
        ecg_ep  = normalize_signal(ecg_ep)
        spo2_ep = normalize_signal(spo2_ep)
        resp_ep = normalize_signal(resp_ep)

        multimodal = np.stack([ecg_ep, spo2_ep, resp_ep], axis=0).astype(np.float32)

        epochs.append({
            'subject_id': subject_id, 'epoch_idx': ep_idx,
            'ecg': ecg_ep.astype(np.float32), 'multimodal': multimodal,
            'label': int(label), 'ahi': np.float32(ahi_estimate),
            'sqi': np.float32(sqi), 'clinical': clinical_feat, 'dataset': 'ucddb',
        })
    return epochs

In [ ]:
import os
import glob
import pandas as pd
from tqdm.auto import tqdm

xls_path = os.path.join(UCDDB_DIR, 'SubjectDetails.xls')
clinical_df = read_subject_details(xls_path) if os.path.exists(xls_path) else None

ucddb_subjects = sorted(set(
    os.path.splitext(os.path.basename(f))[0]
    for f in glob.glob(os.path.join(UCDDB_DIR, 'ucddb*.rec'))
))

ucddb_data = []
for subj in tqdm(ucddb_subjects, desc='UCDDB Subjects'):
    epochs = load_ucddb_subject(subj, UCDDB_DIR, clinical_df)
    ucddb_data.extend(epochs)
    if epochs:
        pos = sum(e['label'] for e in epochs)
        tqdm.write(f'  {subj}: {len(epochs)} epochs | Apnea: {pos} ({pos/len(epochs)*100:.0f}%)')

if ucddb_data:
    pos_ucddb = sum(e['label'] for e in ucddb_data)
    print(f'\nUCDDB Summary: {len(ucddb_data)} epochs | Apnea: {pos_ucddb}')

UCDDB Subjects:   0%|          | 0/25 [00:00<?, ?it/s]

  ucddb002: 749 epochs | Apnea: 74 (10%)
  ucddb003: 882 epochs | Apnea: 167 (19%)
  ucddb005: 826 epochs | Apnea: 31 (4%)
  ucddb006: 808 epochs | Apnea: 37 (5%)
  ucddb007: 813 epochs | Apnea: 46 (6%)
  ucddb008: 768 epochs | Apnea: 1 (0%)
  ucddb009: 925 epochs | Apnea: 41 (4%)
  ucddb010: 907 epochs | Apnea: 59 (7%)
  ucddb011: 901 epochs | Apnea: 17 (2%)
  ucddb012: 864 epochs | Apnea: 124 (14%)
  ucddb013: 811 epochs | Apnea: 37 (5%)
  ucddb014: 774 epochs | Apnea: 164 (21%)
  ucddb015: 916 epochs | Apnea: 19 (2%)
  ucddb017: 789 epochs | Apnea: 29 (4%)
  ucddb018: 822 epochs | Apnea: 5 (1%)
  ucddb019: 852 epochs | Apnea: 30 (4%)
  ucddb020: 752 epochs | Apnea: 36 (5%)
  ucddb021: 913 epochs | Apnea: 22 (2%)
  ucddb022: 788 epochs | Apnea: 9 (1%)
  ucddb023: 861 epochs | Apnea: 86 (10%)
  ucddb024: 908 epochs | Apnea: 81 (9%)
  ucddb025: 711 epochs | Apnea: 314 (44%)
  ucddb026: 838 epochs | Apnea: 45 (5%)
  ucddb027: 893 epochs | Apnea: 130 (15%)
  ucddb028: 722 epochs | Apnea:

In [ ]:
import pickle
import os
import numpy as np

def normalize_clinical_features(data_list):
    """Fit-transform clinical features across all epochs."""
    clinical_arr = np.stack([e['clinical'] for e in data_list])  # [N, 5]
    mean = clinical_arr.mean(axis=0)
    std  = clinical_arr.std(axis=0) + 1e-8

    for e in data_list:
        e['clinical'] = ((e['clinical'] - mean) / std).astype(np.float32)

    return mean, std

# Normalize clinical features if UCDDB data exists
if 'ucddb_data' in locals() and ucddb_data:
    clin_mean, clin_std = normalize_clinical_features(ucddb_data)
    print('Clinical feature stats (before normalization):')
    feat_names = ['age', 'gender', 'bmi', 'neck_circ', 'ahi_history']
    for i, name in enumerate(feat_names):
        print(f'  {name:<15}: mean={clin_mean[i]:.2f}, std={clin_std[i]:.2f}')
    print('\nClinical features normalized successfully.')

# Save the datasets
physionet_out = os.path.join(OUTPUT_DIR, 'physionet_dataset.pkl')
ucddb_out     = os.path.join(OUTPUT_DIR, 'ucddb_dataset.pkl')

if 'physionet_data' in locals() and physionet_data:
    with open(physionet_out, 'wb') as f:
        pickle.dump(physionet_data, f)
    print(f'Saved PhysioNet dataset to {physionet_out}')

if 'ucddb_data' in locals() and ucddb_data:
    with open(ucddb_out, 'wb') as f:
        pickle.dump(ucddb_data, f)
    print(f'Saved UCDDB dataset to {ucddb_out}')

print('\n=== Preprocessing Complete ===')

Clinical feature stats (before normalization):
  age            : mean=49.75, std=9.45
  gender         : mean=0.84, std=0.37
  bmi            : mean=31.56, std=3.87
  neck_circ      : mean=38.00, std=0.00
  ahi_history    : mean=23.86, std=19.29

Clinical features normalized successfully.
Saved PhysioNet dataset to /content/processed/physionet_dataset.pkl
Saved UCDDB dataset to /content/processed/ucddb_dataset.pkl

=== Preprocessing Complete ===


In [ ]:
# ============================================================
# CELL 8: Save Processed Datasets
# ============================================================

physionet_out = os.path.join(OUTPUT_DIR, 'physionet_dataset.pkl')
ucddb_out     = os.path.join(OUTPUT_DIR, 'ucddb_dataset.pkl')

with open(physionet_out, 'wb') as f:
    pickle.dump(physionet_data, f)
print(f'PhysioNet saved: {physionet_out} ({len(physionet_data)} epochs)')

with open(ucddb_out, 'wb') as f:
    pickle.dump(ucddb_data, f)
print(f'UCDDB saved    : {ucddb_out} ({len(ucddb_data)} epochs)')

# Also save clinical normalization stats
meta = {
    'clinical_mean' : clin_mean.tolist() if ucddb_data else None,
    'clinical_std'  : clin_std.tolist()  if ucddb_data else None,
    'feature_names' : ['age', 'gender', 'bmi', 'neck_circ', 'ahi_history'],
    'target_fs'     : TARGET_FS,
    'epoch_duration': EPOCH_DURATION,
    'ecg_samples'   : ECG_SAMPLES,
}
import json
with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('Metadata saved.')

print('\n=== Preprocessing Complete ===')
print(f'PhysioNet: {len(physionet_data)} epochs from {len(set(e["subject_id"] for e in physionet_data))} subjects')
print(f'UCDDB    : {len(ucddb_data)} epochs from {len(set(e["subject_id"] for e in ucddb_data))} subjects')

PhysioNet saved: /content/processed/physionet_dataset.pkl (34053 epochs)
UCDDB saved    : /content/processed/ucddb_dataset.pkl (20793 epochs)
Metadata saved.

=== Preprocessing Complete ===
PhysioNet: 34053 epochs from 35 subjects
UCDDB    : 20793 epochs from 25 subjects


In [ ]:
import shutil
import os

# Define the target Google Drive backup directory
drive_backup_dir = '/content/drive/MyDrive/SA ECG dataset'
os.makedirs(drive_backup_dir, exist_ok=True)

# Define local processed source directory
local_processed_dir = '/content/processed'

if os.path.exists(local_processed_dir):
    for file_name in os.listdir(local_processed_dir):
        src_file = os.path.join(local_processed_dir, file_name)
        dst_file = os.path.join(drive_backup_dir, file_name)
        shutil.copy(src_file, dst_file)
    print(f'Successfully copied processed datasets to Google Drive: {drive_backup_dir}')
    print('Files in Drive folder:', os.listdir(drive_backup_dir))
else:
    print(f'Source local directory {local_processed_dir} not found!')

Successfully copied processed datasets to Google Drive: /content/drive/MyDrive/SA ECG dataset
Files in Drive folder: ['apnea-ecg-database-1.0.0.zip', 'st-vincents-university-hospital-university-college-dublin-sleep-apnea-database-1.0.0.zip', 'physionet_dataset.pkl', 'ucddb_dataset.pkl', 'metadata.json']
